# v0.15.0 -- `call_function()`: calling custom `fn::` stored functions

SurrealDB lets you declare server-side functions with `DEFINE FUNCTION fn::...`. v0.15.0 makes
them callable from the ORM, with arguments **bound as query parameters**, an optional typed
result, and correct behaviour inside a transaction.

```python
acquired = await SurrealDBConnectionManager.call_function(
    "fn::acquire_lock", [table_id, pod_id, 30],
)
```

**On both DB lines**, the call itself behaves identically -- SurrealDB 2.6.x and 3.x agree. The
one divergence is `tx=`, and it is the v0.9.0 transaction contract, not something new: an
*interactive* transaction (WebSocket + SurrealDB 3.x) hands back the function's value right away,
while a *buffered* one (SurrealDB 2.6.x or HTTP) queues the call and returns `None` until commit.
This notebook probes the server and demonstrates whichever path applies, so it runs top to bottom
on either line.

> **Note**: the SDK 2.0 has no `run()`/`call()` method, so the ORM builds the call itself and
> sends it through `query()`. That detail turns out to matter -- see section 6.

## 1. Connect

WebSocket (`.../rpc`). Point `SURREALDB_HOST` / `SURREALDB_PORT` at your own server to run this
notebook elsewhere.

In [1]:
import contextlib
import os

from surreal_orm_lite import BaseSurrealModel, SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Declare the demo functions

Declaring a function is **DDL**, so it goes through `query()` -- exactly how you would create one
today. (A `define_function()` helper is planned for v0.31.0, alongside the other `schema.py` DDL
helpers; v0.15.0 is about *calling*, not *declaring*.)

`OVERWRITE` makes this cell idempotent, so the notebook can be re-executed in place.

In [2]:
client = await SurrealDBConnectionManager.get_client()

FUNCTIONS = {
    "greet": "DEFINE FUNCTION OVERWRITE fn::greet($name: string) { RETURN 'Hello, ' + $name; };",
    # Deliberately NOT commutative: a wrong argument order would give a wrong answer.
    "discount": (
        "DEFINE FUNCTION OVERWRITE fn::discount($price: float, $percent: float) "
        "{ RETURN $price - ($price * $percent / 100); };"
    ),
    "billing::summary": (
        "DEFINE FUNCTION OVERWRITE fn::billing::summary($cart: string, $total: float) "
        "{ RETURN { cart: $cart, total: $total, currency: 'CAD' }; };"
    ),
    "reserve": (
        "DEFINE FUNCTION OVERWRITE fn::reserve($seats: int) "
        "{ UPDATE stock:main SET remaining -= $seats; };"
    ),
}

for statement in FUNCTIONS.values():
    await client.query(statement, {})

await client.query("UPSERT stock:main SET remaining = 10;", {})
print("Defined:", ", ".join(f"fn::{name}" for name in FUNCTIONS))

Defined: fn::greet, fn::discount, fn::billing::summary, fn::reserve


## 3. Positional arguments

SurrealQL function arguments are **positional**, so the plain form takes a sequence in
declaration order. The `fn::` prefix is optional, and nested namespaces work.

In [3]:
print(await SurrealDBConnectionManager.call_function("fn::greet", ["Ada"]))

# The fn:: prefix is added when absent
print(await SurrealDBConnectionManager.call_function("greet", ["Grace"]))

# Nested namespace
print(await SurrealDBConnectionManager.call_function("fn::billing::summary", ["cart-1", 42.5]))

# 100 - 10% = 90.0
print(await SurrealDBConnectionManager.call_function("fn::discount", [100.0, 10.0]))

Hello, Ada
Hello, Grace
{'cart': 'cart-1', 'currency': 'CAD', 'total': 42.5}
90.0


### Arguments are bound, never interpolated

Only the function *name* is written into the statement (SurrealQL accepts no bound parameter in
call position), and it is validated as an identifier path first. Every **value** is bound, so a
hostile string is data and nothing else.

In [4]:
hostile = "');DELETE stock;--"
print(await SurrealDBConnectionManager.call_function("fn::greet", [hostile]))

remaining = await client.query("SELECT * FROM stock:main;", {})
print("stock table intact:", remaining)

# An invalid *name* is rejected before any query is issued.
try:
    await SurrealDBConnectionManager.call_function("fn::greet; DROP TABLE stock")
except ValueError as exc:
    print("Rejected:", exc)

Hello, ');DELETE stock;--
stock table intact: [{'id': RecordID(table_name=stock, record_id='main'), 'remaining': 10}]
Rejected: Invalid stored function name: 'fn::greet; DROP TABLE stock'. Expected 'fn::name' or 'fn::namespace::name', where each segment is an identifier ([A-Za-z_][A-Za-z0-9_]*).


## 4. Named arguments with `params=`

Positional arguments are easy to transpose silently. `params=` lets you name them: the ORM reads
the function's declared signature from `INFO FOR DB` (cached per namespace/database) and orders
them for you, so **the mapping's own order is irrelevant**.

In [5]:
# Same call, three orderings -- all correct, because the signature decides.
for mapping in (
    {"price": 100.0, "percent": 10.0},
    {"percent": 10.0, "price": 100.0},
):
    value = await SurrealDBConnectionManager.call_function("fn::discount", params=mapping)
    print(mapping, "->", value)

# A key that does not match the declaration is caught, and the error names what was expected.
try:
    await SurrealDBConnectionManager.call_function("fn::discount", params={"price": 100.0, "pct": 10.0})
except ValueError as exc:
    print("Rejected:", exc)

# args and params are two ways to say the same thing; passing both is ambiguous.
try:
    await SurrealDBConnectionManager.call_function("fn::discount", [100.0, 10.0], params={"price": 1.0, "percent": 2.0})
except ValueError as exc:
    print("Rejected:", exc)

{'price': 100.0, 'percent': 10.0} -> 90.0
{'percent': 10.0, 'price': 100.0} -> 90.0
Rejected: Arguments for 'fn::discount' do not match its declared parameters. Expected ['price', 'percent'], got ['pct', 'price'].
Rejected: Pass either 'args' (positional) or 'params' (named), not both — they are two ways to supply the same arguments.


## 5. Typed results with `return_type=`

`return_type=` accepts anything Pydantic can adapt -- a model, a dataclass, a scalar, or a generic
like `list[Model]` -- so there is one rule to remember rather than one per shape.

Note this is Pydantic **validation**, not a cast: an `int` asked to be a `str` is a mismatch, not
a silent `str(5)`.

In [6]:
from pydantic import BaseModel

from surreal_orm_lite.exceptions import SurrealDbNotFoundError, SurrealDbValidationError


class CartSummary(BaseModel):
    cart: str
    total: float
    currency: str


summary = await SurrealDBConnectionManager.call_function(
    "fn::billing::summary", ["cart-1", 42.5], return_type=CartSummary,
)
print(type(summary).__name__, "->", summary)
print("total * 2 =", summary.total * 2)

# A result that does not fit the requested type is a loud error, not a surprise later on.
try:
    await SurrealDBConnectionManager.call_function("fn::greet", ["Ada"], return_type=CartSummary)
except SurrealDbValidationError as exc:
    print("Rejected:", str(exc).splitlines()[0])

# A function the server does not know is normalised too.
try:
    await SurrealDBConnectionManager.call_function("fn::not_defined_anywhere")
except SurrealDbNotFoundError as exc:
    print("Rejected:", exc)

CartSummary -> cart='cart-1' total=42.5 currency='CAD'
total * 2 = 85.0
Rejected: The value returned by 'fn::greet' does not fit <class '__main__.CartSummary'>: 1 validation error for CartSummary
Rejected: Stored function 'fn::not_defined_anywhere' does not exist: Function 'fn::not_defined_anywhere' not found: The function 'fn::not_defined_anywhere' does not exist


## 6. Inside a transaction

Pass `tx=` so the function runs **inside** the transaction. Without it the call would execute
outside the open transaction and silently break atomicity -- which matters, because stored
functions typically mutate state (locks, counters, stock levels).

This is also where the SDK's missing `run()` method becomes visible. The ORM emits the **bare
call form** `fn::reserve($_fnarg0);` rather than `RETURN fn::reserve(...)`. The reason is
empirical: a `RETURN` inside a `BEGIN ... COMMIT` batch terminates the transaction early *and
silently* -- every statement queued after it is reported `status: OK` and never executes, on both
SurrealDB 2.6.5 and 3.2.4. The cell below queues a `save()` *after* the call precisely to show
that it still lands.

In [7]:
class Booking(BaseSurrealModel):
    id: str
    seats: int = 0


with contextlib.suppress(Exception):
    await client.query("DELETE Booking;", {})
await client.query("UPSERT stock:main SET remaining = 10;", {})

async with SurrealDBConnectionManager.transaction() as tx:
    await SurrealDBConnectionManager.call_function("fn::reserve", [3], tx=tx)
    await Booking(id="b1", seats=3).save(tx=tx)

print("stock after commit:", await client.query("SELECT remaining FROM stock:main;", {}))
print("booking queued AFTER the call:", await client.query("SELECT * FROM Booking;", {}))

stock after commit: [{'remaining': 7}]
booking queued AFTER the call: [{'id': RecordID(table_name=Booking, record_id='b1'), 'seats': 3}]


### Rollback

An exception in the block rolls the whole thing back -- the function's own writes included.

In [8]:
await client.query("UPSERT stock:main SET remaining = 10;", {})

with contextlib.suppress(RuntimeError):
    async with SurrealDBConnectionManager.transaction() as tx:
        await SurrealDBConnectionManager.call_function("fn::reserve", [4], tx=tx)
        raise RuntimeError("something went wrong downstream")

print("stock after rollback:", await client.query("SELECT remaining FROM stock:main;", {}))

stock after rollback: [{'remaining': 10}]


### The one behaviour that differs between DB lines

`transaction()` picks its strategy from what the server supports. Probe it, then read the branch
that applies to you -- both are shown so the notebook runs on either line.

In [9]:
async def interactive_transactions_supported() -> bool:
    """True if the server exposes the SDK's native transaction RPC (SurrealDB 3.x)."""
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await interactive_transactions_supported()
print("Interactive transactions available (SurrealDB 3.x + WebSocket):", interactive)

async with SurrealDBConnectionManager.transaction() as tx:
    value = await SurrealDBConnectionManager.call_function("fn::greet", ["Ada"], tx=tx)
    if tx.is_interactive:
        print("Interactive: the value is available immediately ->", repr(value))
    else:
        print("Buffered: the call is queued, so nothing is returned yet ->", repr(value))
        print("The function still runs -- atomically, at commit.")

Interactive transactions available (SurrealDB 3.x + WebSocket): True
Interactive: the value is available immediately -> 'Hello, Ada'


In [10]:
# Asking for a typed result from a buffered transaction is a contradiction: there is no value
# yet to coerce. The ORM says so instead of handing back a silent None.
async with SurrealDBConnectionManager.transaction() as tx:
    if tx.is_interactive:
        typed = await SurrealDBConnectionManager.call_function(
            "fn::billing::summary", ["cart-9", 10.0], tx=tx, return_type=CartSummary,
        )
        print("Interactive + return_type ->", typed)
    else:
        try:
            await SurrealDBConnectionManager.call_function(
                "fn::billing::summary", ["cart-9", 10.0], tx=tx, return_type=CartSummary,
            )
        except ValueError as exc:
            print("Buffered + return_type -> rejected:", exc)

Interactive + return_type -> cart='cart-9' total=10.0 currency='CAD'


## 7. Calling from a model

A stored function is not bound to a table, so `Model.call_function()` is a pure convenience for
code organised around models -- it delegates verbatim.

In [11]:
print(await Booking.call_function("fn::greet", ["Ada"]))
print(await Booking.call_function("fn::discount", params={"percent": 25.0, "price": 80.0}))

Hello, Ada
60.0


## 8. Cleanup

In [12]:
for name in FUNCTIONS:
    with contextlib.suppress(Exception):
        await client.query(f"REMOVE FUNCTION fn::{name};", {})

for table in ("Booking", "stock"):
    with contextlib.suppress(Exception):
        await client.query(f"REMOVE TABLE {table};", {})

await SurrealDBConnectionManager.close_connection()
print("Cleaned up.")

Cleaned up.
